In [1]:
# -*- coding: utf-8 -*-
"""
粒化学习 · 点图数据起手包（纯 NumPy，无黑箱）
=============================================
用途：给"粒化 / 粒球"的动手实验准备二维点集，能立刻 scatter 出来看。

为什么先从这个文件开始：
  1) 数据不是障碍，点集 = 一个 (n, 2) 的 float 数组。真正的选择是"什么样的点集
     能让'多粗才够(Q3)'这个问题变得可判"。
  2) 全部用 NumPy 手写生成器 → 每个点的来历你都知道，可以精确控制难度
     （分离度、密度差、离群点、层次深度），而不是拿一个不透明的数据集。
  3) 生成器同时返回真值标签 y，用于你后面的"三栏对照"校准，而不是用来判罪。

跑法：
    python granulation_data_starters.py
输出：
    data_overview.png      2x3 总览图（每组一栏）
    *_points.npz           每组点集的 X / y，便于复现实验
"""

import numpy as np

# ============================================================
# 一、生成器区（纯 NumPy，全部返回 (X, y)，X.shape = (n, 2)）
# ============================================================


def uniform_box(n=400, seed=0, lo=-1.0, hi=1.0):
    """均匀随机点。故意作为反例：它没有结构，任何 k 都同样"对"，
    Q3（多粗才够）在这里是不可判的。"""
    rng = np.random.default_rng(seed)
    X = rng.uniform(lo, hi, size=(n, 2))
    return X, np.zeros(n, dtype=int)


def gaussian_blobs(ns, centers=None, sigmas=None, seed=0):
    """高斯混合：k 个团，每团点数 ns[i]、中心 centers[i]、标准差 sigmas[i]。
    把 sigmas 设成差别很大，就得到"密度悬殊"的点集。"""
    rng = np.random.default_rng(seed)
    ns = np.asarray(ns, dtype=int)
    k = len(ns)
    if centers is None:
        centers = rng.uniform(-0.8, 0.8, size=(k, 2))
    centers = np.asarray(centers, dtype=float)
    if sigmas is None:
        sigmas = [0.09] * k
    if np.isscalar(sigmas):
        sigmas = [float(sigmas)] * k

    X, y = [], []
    for i in range(k):
        X.append(centers[i] + sigmas[i] * rng.standard_normal((ns[i], 2)))
        y.append(np.full(ns[i], i))
    return np.vstack(X), np.concatenate(y)


def add_outliers(X, y, m=12, r_in=1.15, r_out=1.45, seed=0):
    """在数据外圈撒 m 个离群点（标签记 -1）。用来测"鲁棒性":
    一个粗粒该不该把这几个点吃进去？覆盖不足和过度覆盖哪个代价大？"""
    rng = np.random.default_rng(seed)
    th = rng.uniform(0, 2 * np.pi, m)
    r = rng.uniform(r_in, r_out, m)
    Xo = np.c_[r * np.cos(th), r * np.sin(th)]
    return np.vstack([X, Xo]), np.concatenate([y, -np.ones(m, dtype=int)])


def two_moons(n=400, noise=0.05, seed=0):
    """两个相互缠绕的半圆（非凸）。经典陷阱：
    任何"以全局距离/全局密度"为判据的粒化都会在这里翻车。"""
    rng = np.random.default_rng(seed)
    n1 = n // 2
    n2 = n - n1
    t1 = np.pi * rng.uniform(0, 1, n1)
    t2 = np.pi * rng.uniform(0, 1, n2)
    m1 = np.c_[np.cos(t1), np.sin(t1)]
    m2 = np.c_[1 - np.cos(t2), 0.5 - np.sin(t2)]
    X = np.vstack([m1, m2]) + noise * rng.standard_normal((n, 2))
    y = np.r_[np.zeros(n1, dtype=int), np.ones(n2, dtype=int)]
    return X, y


def rings(radii=(0.45, 1.0), widths=(0.035, 0.13), counts=(150, 250), seed=0):
    """同心环，且内外环带宽（密度）不同。
    用来试"什么时候该说'我不分了'"：内环细、外环散，
    一个判据若只认密度，就会把内环切成很多小粒、外环合成一个大粒。"""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for i, (r0, w, n) in enumerate(zip(radii, widths, counts)):
        th = rng.uniform(0, 2 * np.pi, n)
        r = r0 + w * rng.standard_normal(n)
        X.append(np.c_[r * np.cos(th), r * np.sin(th)])
        y.append(np.full(n, i))
    return np.vstack(X), np.concatenate(y)


def two_spirals(n=400, turns=1.6, noise=0.04, seed=0):
    """双螺旋：两类互为 180° 旋转（不是镜像）。两条臂在旋转方向上交替贴近，
    “局部判据”与“全局判据”的差别在这里被放到最大。"""
    rng = np.random.default_rng(seed)
    n1 = n // 2
    n2 = n - n1

    def arm(n_, rot):
        u = rng.uniform(0, 1, n_)
        r = 0.08 + 0.92 * np.sqrt(u)  # 半径上均匀（面积均匀）
        t = turns * 2 * np.pi * u  # 螺旋参数
        return np.c_[r * np.cos(t + rot), r * np.sin(t + rot)]

    X = np.vstack([arm(n1, 0.0), arm(n2, np.pi)])  # 第二类整体转 π
    X = X + noise * rng.standard_normal((n, 2))
    y = np.r_[np.zeros(n1, dtype=int), np.ones(n2, dtype=int)]
    return X, y


def hierarchical(
    n_big=3, n_sub=35, spread_big=0.16, spread_sub=0.045, seed=0, centers=None
):
    """层次嵌套：n_big 个大簇，每个大簇里再含 3 个小簇。
    这是 Q3 的试金石：同一片点，粗粒下是 3 个、细粒下是 9 个，
    两个答案都"对"。返回两级标签 y_big / y_sub，可用于事后校准。"""
    rng = np.random.default_rng(seed)
    if centers is None:
        centers = np.array([[-0.85, 0.60], [0.85, 0.50], [0.0, -0.90]])
    X, y_big, y_sub = [], [], []
    for b in range(n_big):
        for s in range(3):
            c = centers[b] + spread_big * rng.standard_normal(2)
            X.append(c + spread_sub * rng.standard_normal((n_sub, 2)))
            y_big.append(np.full(n_sub, b))
            y_sub.append(np.full(n_sub, b * 3 + s))
    return np.vstack(X), np.concatenate(y_big), np.concatenate(y_sub)


# ============================================================
# 二、拿现成的二维真数据
# ============================================================


def load_libsvm_2d(path, take=2):
    """读 LIBSVM 格式（label idx:val ...），只取前 take 列。
    fourclass 数据集就是 862x2，粒球/GBC 类论文画二维粒化图常用它。
    下载：https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary.html
          （找 fourclass，862 instances / 2 features）"""
    rows, ys = [], []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue
            ys.append(float(parts[0]))
            d = {}
            for kv in parts[1:]:
                k, v = kv.split(":")
                d[int(k)] = float(v)
            rows.append([d.get(i + 1, 0.0) for i in range(take)])
    X = np.asarray(rows, dtype=float)
    y = np.asarray(ys)
    _, y = np.unique(y, return_inverse=True)  # 标签重排成 0..k-1
    return X, y


def pca_2d(X, whiten=False):
    """把任意高维数据降到 2 维 —— 纯 NumPy，用协方差矩阵的特征分解。
    （比 SVD 快得多：对 784 维特征只需分解 784x784 的矩阵。）
    返回 (Z, explained_ratio)。"""
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=0)
    Xc = X - mu
    C = (Xc.T @ Xc) / (len(Xc) - 1)  # 特征协方差
    w, V = np.linalg.eigh(C)  # 升序
    order = np.argsort(w)[::-1][:2]
    Z = Xc @ V[:, order]
    ratio = w[order].sum() / w.sum()
    if whiten:
        Z = Z / np.sqrt(w[order] + 1e-12)
    return Z, float(ratio)


def standardize(X):
    """列标准化。做 PCA / 算距离前，量纲不同必须先标准化。"""
    X = np.asarray(X, dtype=float)
    return (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)


# ============================================================
# 三、总览绘图
# ============================================================


def plot_all(datasets, path="data_overview.png"):
    """datasets: [(标题, X, y, 备注), ...]"""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False

    n = len(datasets)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.6 * rows))
    axes = np.atleast_1d(axes).ravel()
    cmap = plt.get_cmap("tab10")

    for ax, (title, X, y, note) in zip(axes, datasets):
        for lab in np.unique(y):
            m = y == lab
            color = "#111111" if lab == -1 else cmap(int(lab) % 10)
            size = 6 if lab == -1 else 12
            ax.scatter(X[m, 0], X[m, 1], s=size, c=[color], linewidths=0, alpha=0.8)
        ax.set_title(f"{title}  (n={len(X)})", fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal", adjustable="datalim")
        if note:
            ax.set_xlabel(note, fontsize=8.5, color="#666666")

    for ax in axes[len(datasets) :]:
        ax.axis("off")

    fig.tight_layout()
    fig.savefig(path, dpi=150, facecolor="white")
    print(f"[saved] {path}")


# ============================================================
# 四、主流程
# ============================================================


def build_all(seed=0):
    sets = []

    X, y = uniform_box(400, seed=seed)
    sets.append(("① 均匀随机（无结构）", X, y, "任何 k 都同样“对” → Q3 不可判"))

    X, y = gaussian_blobs(
        [120] * 4,
        centers=[[-0.8, 0.6], [0.8, 0.7], [-0.7, -0.7], [0.75, -0.6]],
        sigmas=0.09,
        seed=seed,
    )
    sets.append(("② 高斯混合·等密度", X, y, "标准情形，检验基本功"))

    X, y = gaussian_blobs(
        [150, 150, 150],
        centers=[[-0.75, 0.5], [0.8, 0.45], [0.0, -0.85]],
        sigmas=[0.05, 0.11, 0.22],
        seed=seed,
    )
    X, y = add_outliers(X, y, m=14, seed=seed + 100)
    sets.append(("③ 密度悬殊 + 离群点", X, y, "粗粒该不该吃掉离群点"))

    X, y = two_moons(400, noise=0.05, seed=seed)
    sets.append(("④ 两个月亮（非凸缠绕）", X, y, "局部判据 vs 全局距离"))

    X, y = rings(seed=seed)
    sets.append(("⑤ 同心环（内外密度不同）", X, y, "什么时候该说“我不分了”"))

    X, y_big, y_sub = hierarchical(seed=seed)
    sets.append(("⑥ 层次嵌套（大簇含小簇）", X, y_big, "多粒度的试金石：Q3 的核心"))
    sets.append(("⑥b 同上·细粒度真值", X, y_sub, "同一片点，粗/细两个答案都“对”"))

    X, y = two_spirals(seed=seed)
    sets.append(("⑦ 双螺旋", X, y, "两类互为 180° 旋转，交替贴近"))

    return sets


if __name__ == "__main__":
    from pathlib import Path

    # 产出目录 = 与数据放在一起（1-data/）。
    # 注意：Jupyter 的当前目录是「启动 Jupyter 的那个目录」，不一定等于 notebook 所在目录。
    # 所以这里显式判定一下，免得在项目根启动 Jupyter 时，9 个 npz 全甩到根目录上去。
    OUTDIR = Path.cwd()
    if (OUTDIR / "1-data").is_dir():  # 说明是在项目根启动的
        OUTDIR = OUTDIR / "1-data"
    OUTDIR.mkdir(parents=True, exist_ok=True)
    print(f"产出目录：{OUTDIR.resolve()}")

    sets = build_all(seed=0)
    for i, (title, X, y, note) in enumerate(sets):
        print(f"{i:02d} {title:26s} X={str(X.shape):12s} 类别数={len(np.unique(y))}")
        np.savez(OUTDIR / f"points_{i:02d}.npz", X=X, y=y)
    plot_all(sets, OUTDIR / "data_overview.png")

    # —— 附加：把 MNIST 像素降到 2 维，得到真实的"点图" ——
    # X_train 形如 (60000, 784)，先标准化再 PCA：
    #   from your_mnist_loader import load_mnist
    #   Xtr, ytr = load_mnist()
    #   Z, ratio = pca_2d(standardize(Xtr))
    #   print("前两主成分解释方差比：", ratio)
    #   然后 scatter(Z[:,0], Z[:,1], c=ytr, s=2)
    # 注意：2D 投影丢信息，2D 图上看到的粒化结果 ≠ 原空间的结果。


00 ① 均匀随机（无结构）                X=(400, 2)     类别数=1
01 ② 高斯混合·等密度                 X=(480, 2)     类别数=4
02 ③ 密度悬殊 + 离群点               X=(464, 2)     类别数=4
03 ④ 两个月亮（非凸缠绕）               X=(400, 2)     类别数=2
04 ⑤ 同心环（内外密度不同）              X=(400, 2)     类别数=2
05 ⑥ 层次嵌套（大簇含小簇）              X=(315, 2)     类别数=3
06 ⑥b 同上·细粒度真值                X=(315, 2)     类别数=9
07 ⑦ 双螺旋                      X=(400, 2)     类别数=2
[saved] data_overview.png
